In [ ]:
from supabase import create_client
from dotenv import load_dotenv
import os
import pandas as pd
import plotly.express as px
from datetime import datetime

import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.getLogger("httpx").setLevel(logging.INFO) # Supabase API logger

logger = logging.getLogger(__name__)

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

default_order_columns = {
    "customers": "customer_id"
}

# Here's yet another proof why SQLAlchemy would be much more efficient and flexible tool for data loading.
# Using Supabase library only to follow the SHU/HA exercises in the weekly workbook.
def load_data_over_api(table, select="*", filterer=None, sorter=None):
    all_rows = []
    start = 0
    step = 1000 # Supabase API sets a limit to the number of rows returned
    default_order_column = default_order_columns.get(table, "id")
    
    while True:
        # Unfiltered API call.
        api_call = supabase.table(table).select(select)

        # Apply optional filters using filterer function.
        if filterer:
            api_call = filterer(api_call)

        # Add optional sorting.
        if sorter:
            api_call = sorter(api_call)
        
        # Always sort by default column after custom sortings to ensure determinism during batching.
        api_call = api_call.order(default_order_column)

        # Apply range.
        # Supabase range() is inclusive, unlike Python range. That's why -1 is needed for the second argument.
        api_call = api_call.range(start, start + step - 1)
        
        # Execute API call.
        response = api_call.execute()
        
        batch = response.data
        all_rows.extend(batch)

        # Check if last batch.
        if len(batch) < step:
            break
        
        start += step

    # When joining tables over Supabase API, the joined rows will appear as JSON columns in result.
    # We need to flatten such columns so that the final result would be similar to a result which we would get using pd.merge().
    df = pd.DataFrame(all_rows)
    return flatten_dataframe_with_json(df, detect_json_columns(df))

def detect_json_columns(df):
    json_cols = []

    for col in df.columns:
        # 1. Drop missing values to inspect real data types safely
        non_empty_rows = df[col].dropna()
        
        # 2. Skip the column if it has no data at all
        if non_empty_rows.empty:
            continue
            
        # 3. Inspect the very first available value in the column
        first_value = non_empty_rows.iloc[0]
        
        # 4. If the value is a dictionary (JSON object) or a list, flag it for flattening
        if isinstance(first_value, (dict, list)):
            json_cols.append(col)

    return json_cols

def flatten_dataframe_with_json(df, json_columns):
    result = df
    for column in json_columns:
        # Check if the column still exists in the current result DataFrame
        if column in result.columns:
            # Flatten the nested JSON structure into a separate DataFrame
            flattened_column = pd.json_normalize(result[column])
            
            # Add a prefix to the new columns (e.g., "customers.name") 
            # to prevent column name collisions with the main table
            flattened_column = flattened_column.add_prefix(f"{column}.")
            
            # Drop the original JSON column and concatenate the flattened columns horizontally
            result = pd.concat([result.drop(column, axis=1), flattened_column], axis=1)
            
    return result

# Loading unfiltered sales data.
df_orders = load_data_over_api("sales")
df_orders["sale_date"] = pd.to_datetime(df_orders["sale_date"])

# Loading unfiltered customers data.
df_customers = load_data_over_api("customers")

df_merged = pd.merge(df_orders, df_customers, how="left")

print(f"Orders: {len(df_orders)}, Customers: {len(df_customers)}")
df_merged.dtypes

In [ ]:
# Adding filters and sorting would make using Supabase API even more complex while trying to bypass the 1000 row limit.
df_customers_tallinn = load_data_over_api("customers", filterer=lambda api_call: api_call.eq("city", "Tallinn"))
df_orders_sorted = load_data_over_api("sales", sorter=lambda api_call: api_call.order("total_price", desc=True))
df_customers_tallinn.shape, df_orders_sorted.shape

In [ ]:
df_orders_sorted.head()

In [ ]:
df_tallinn = load_data_over_api(
    "sales",
    select="*, customers!inner(*)", # INNER JOIN customers and select all columns
    filterer=lambda api_call: api_call.eq("customers.city", "Tallinn"),
    sorter=lambda api_call: api_call.order("total_price", desc=True)
)

df_tallinn.head()

In [ ]:
df_tallinn.shape

In [ ]:
# Let's try another city.
df_tartu = load_data_over_api(
    "sales",
    select="*, customers!inner(*)", # INNER JOIN customers and select all columns
    filterer=lambda api_call: api_call.eq("customers.city", "Tartu"),
    sorter=lambda api_call: api_call.order("total_price", desc=True)
)

df_tartu.head()

In [ ]:
df_tartu.shape

In [ ]:
# Aggregations
agg_tartu = pd.DataFrame([{
    "total_orders": df_tartu["id"].count(),
    "total_revenue": df_tartu["total_price"].sum(),
    "average_order": df_tartu["total_price"].mean()
}])

agg_tartu

In [ ]:
# One more city.
df_parnu = load_data_over_api(
    "sales",
    select="*, customers!inner(*)", # INNER JOIN customers and select all columns
    filterer=lambda api_call: api_call.eq("customers.city", "Pärnu"),
    sorter=lambda api_call: api_call.order("total_price", desc=True)
)

df_parnu.head()

In [ ]:
# Aggregations
agg_parnu = pd.DataFrame([{
    "total_orders": df_parnu["id"].count(),
    "total_revenue": df_parnu["total_price"].sum(),
    "average_order": df_parnu["total_price"].mean()
}])

agg_parnu

In [ ]:
date_from = pd.to_datetime("2025-02-01")
date_to = pd.to_datetime("2025-03-01")
df_last_month = load_data_over_api(
    "sales",
    filterer=lambda api_call: api_call.gte("sale_date", date_from).lt("sale_date", date_to),
    sorter=lambda api_call: api_call.order("sale_date")
)
df_last_month.head()

In [ ]:
df_last_month.shape

In [ ]:
df_last_month.describe

In [ ]:
def city_report(df, city):
    """Generate city-specific revenue report."""
    city_data = df[df["city"] == city]
    return {"city": city, "orders": len(city_data),
            "revenue": city_data["total_price"].sum()}

for city in ["Tallinn", "Tartu", "Pärnu"]:
    r = city_report(df_merged, city)
    print(f"{r["city"]}: {r["orders"]} tellimust, {r["revenue"]:.2f} EUR")

In [ ]:
# Logging and exception handling.
def safe_fetch(supabase_client, table_name):
    """Fetch data with safe error handling."""
    try:
        response = supabase_client.table(table_name).select("*").execute()
        df = pd.DataFrame(response.data)
        if len(df) == 0:
            logger.warning(f"Table '{table_name}' is empty!")
        logger.info(f"Loaded {len(df)} rows from table '{table_name}'")
        return df
    except Exception as e:
        logger.error(f"Error reading table '{table_name}': {e}")
        return pd.DataFrame()  # Return an empty DataFrame on error

In [ ]:
# Test 1 - normal flow (unless unexpected API error occurs).
df_products = safe_fetch(supabase, "products")
df_products.head()

In [ ]:
# Test 2 - exception
df_products = safe_fetch(supabase, "productsd") # wrong table name
df_products.head() # Describe wouldn't work on empty dataframe though.

In [ ]:
def stream_api_to_csv(table, file_path, select="*", filterer=None, sorter=None):
    """
    Fetches data from Supabase in batches of 1000 and writes them directly to a CSV file.
    This keeps the memory footprint minimal.
    """
    start = 0
    step = 1000
    default_order_column = default_order_columns.get(table, "id")
    
    # Delete the old file if it already exists to start fresh
    if os.path.exists(file_path):
        os.remove(file_path)
        
    is_first_batch = True
    
    while True:
        api_call = supabase.table(table).select(select)

        if filterer:
            api_call = filterer(api_call)

        if sorter:
            api_call = sorter(api_call)
        else:
            api_call = api_call.order(default_order_column)

        # Deterministic sorting
        api_call = api_call.order(default_order_column)

        response = api_call.range(start, start + step - 1).execute()
        batch = response.data

        # If no more data is returned, break the loop
        if not batch:
            break
            
        # Convert the current batch into a temporary DataFrame
        batch_df = pd.DataFrame(batch)
        
        # --- DYNAMIC JSON COLUMN DETECTION (For this batch) ---
        json_cols = []
        for col in batch_df.columns:
            non_empty_rows = batch_df[col].dropna()
            if non_empty_rows.empty:
                continue
            if isinstance(non_empty_rows.iloc[0], (dict, list)):
                json_cols.append(col)
                
        # Flatten JSON if any exists in this batch
        if json_cols:
            batch_df = flatten_dataframe_with_json(batch_df, json_cols)

        # --- STREAM TO CSV ---
        if is_first_batch:
            # First batch creates the file and writes the headers (column names)
            batch_df.to_csv(file_path, mode="w", index=False)
            is_first_batch = False
        else:
            # Subsequent batches append to the file WITHOUT headers
            batch_df.to_csv(file_path, mode="a", index=False, header=False)

        print(f"Saved rows {start} to {start + len(batch)}...")

        if len(batch) < step:
            break
        
        start += step
        
    print(f"Done! All data streamed successfully to '{file_path}'.")

In [ ]:
def weekly_sales_report(df, report_date=None):
    """Generate a weekly sales report.

    Args:
        df: DataFrame with sales orders
        report_date: Report date (defaults to today)
    Returns:
        dict: Report summary
    """
    if report_date is None:
        report_date = datetime.now().strftime("%Y-%m-%d")
    return {
        "report_date": report_date,
        "total_orders": len(df),
        "total_revenue": round(df["total_price"].sum(), 2),
        "avg_order": round(df["total_price"].mean(), 2),
    }

# Run
result = weekly_sales_report(df_orders)
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
def calculate_rfm(df, reference_date=None):
    """Calculate RFM scores and segments.

    Args:
        df: DataFrame with orders (columns: customer_id, sale_date, total_price)
        reference_date: Reference date for calculating Recency

    Returns:
        DataFrame: RFM scores and segments for each customer
    """
    if reference_date is None:
        reference_date = pd.to_datetime("today")
    else:
        reference_date = pd.to_datetime(reference_date)

    df["sale_date"] = pd.to_datetime(df["sale_date"])

    # Recency: days since last purchase
    recency = df.groupby("customer_id")["sale_date"].max().reset_index()
    recency.columns = ["customer_id", "last_purchase"]
    recency["recency_days"] = (reference_date - recency["last_purchase"]).dt.days

    # Frequency: number of purchases
    frequency = df.groupby("customer_id").size().reset_index(name="frequency")

    # Monetary: total spending
    monetary = df.groupby("customer_id")["total_price"].sum().reset_index()
    monetary.columns = ["customer_id", "monetary"]

    # Merge together
    rfm = recency[["customer_id", "recency_days"]].merge(
        frequency, on="customer_id"
    ).merge(
        monetary, on="customer_id"
    )

    # Score assignment (simplified)
    rfm["R_score"] = pd.qcut(rfm["recency_days"], q=3, labels=[3, 2, 1]).astype(int)
    rfm["F_score"] = pd.qcut(
        rfm["frequency"].rank(method="first"), q=3, labels=[1, 2, 3]
    ).astype(int)
    rfm["M_score"] = pd.qcut(rfm["monetary"], q=3, labels=[1, 2, 3]).astype(int)
    rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

    # Segmentation
    def assign_segment(score):
        if score >= 8:
            return "VIP Champions"
        elif score >= 6:
            return "Potential"
        elif score >= 4:
            return "At Risk"
        else:
            return "Lost"

    rfm["segment"] = rfm["RFM_score"].apply(assign_segment)
    return rfm

# Test
rfm_result = calculate_rfm(df_orders, reference_date="2024-08-01")
print(rfm_result.sort_values("RFM_score", ascending=False))
print(f"\nSegment distribution:")
print(rfm_result["segment"].value_counts())

In [ ]:
def normalize_date(reference_date=None):
    """Normalize various date inputs into a Pandas Timestamp.

    Args:
        reference_date (str, datetime, date, optional): The date to normalize. 
            Can be a string, datetime object, or date object. 
            Defaults to None, which uses the current date.

    Returns:
        pd.Timestamp: The normalized Pandas datetime object.
    """
    return pd.to_datetime(reference_date or datetime.now().date())

normalize_date(), normalize_date('2025-02-28')

In [ ]:
def find_customers_at_churn_risk(df, reference_date=None, days_threshold=60):
    """Find customers who have not made a purchase for a specified number of days.

    Filters the sales data up to the reference date, calculates the days since
    each customer's last purchase using an optimized aggregation, and returns 
    customers whose inactivity exceeds the given threshold.

    Args:
        df (pd.DataFrame): DataFrame containing sales data.
        reference_date (str, datetime, optional): The point in time from which 
            inactivity is calculated. Defaults to current date.
        days_threshold (int, optional): Minimum number of days since last purchase 
            to classify a customer at risk. Defaults to 60.

    Returns:
        pd.DataFrame: Customers at churn risk with their profile details.
    """
    reference_date = normalize_date(reference_date)
    
    # Filter out future transactions
    filtered_df = df[df["sale_date"] <= reference_date]
    
    # Optimized grouping: group only by ID, aggregate everything else
    recency = filtered_df.groupby("customer_id").agg(
        last_purchase=("sale_date", "max"),
        first_name=("first_name", "first"),
        last_name=("last_name", "first"),
        email=("email", "first"),
        phone=("phone", "first"),
        city=("city", "first")
    ).reset_index()
    
    # Sort by newest purchases
    recency = recency.sort_values("last_purchase", ascending=False)
    
    # Calculate inactivity days
    recency["days_since_last_purchase"] = (reference_date - recency["last_purchase"]).dt.days.astype(int)
    
    # Return at-risk customers
    return recency[recency["days_since_last_purchase"] >= days_threshold]

# find_customers_at_churn_risk: test 1
find_customers_at_churn_risk(df_merged, '2024-12-31')


In [ ]:
# find_customers_at_churn_risk: test 2
find_customers_at_churn_risk(df_merged)

In [ ]:
# find_customers_at_churn_risk: test 3
find_customers_at_churn_risk(df_merged, None, 15)

In [ ]:
# find_customers_at_churn_risk: test 4
find_customers_at_churn_risk(df_merged, '2024-06-30', 30)

In [ ]:
# Simplified ETL Pipeline example.

# === EXTRACT ===
def extract_orders():
    """Simulate fetching data from an API."""
    print("[EXTRACT] Loading...")
    data = {
        "customer_id": [1001, 1002, 1003, 1001, 1002, 1004, 1003, 1001, 1005, 1004,
                        1002, 1003, 1005, 1001, 1006, 1004, 1002, 1007, 1003, 1005],
        "sale_date": pd.date_range("2024-01-15", periods=20, freq="10D"),
        "total_price": [89.99, 45.50, 120.00, 67.30, 55.00, 210.00, 33.50, 145.00,
                        78.00, 92.00, 160.00, 44.00, 88.50, 230.00, 37.00, 175.00,
                        110.00, 65.00, 95.00, 125.00],
        "city": ["Tallinn", "Tartu", "Tallinn", "Tallinn", "Tartu", "Parnu", "Tallinn",
                 "Tallinn", "Tartu", "Parnu", "Tartu", "Tallinn", "Tartu", "Tallinn",
                 "Parnu", "Parnu", "Tartu", "Tallinn", "Tallinn", "Tartu"]
    }
    df = pd.DataFrame(data)
    print(f"[EXTRACT] {len(df)} orders loaded")
    return df

# === TRANSFORM ===
def transform_monthly(df):
    """Calculate monthly report."""
    print("[TRANSFORM] Calculating...")
    monthly = df.groupby(
        [
            df["sale_date"].dt.to_period("M"),
            "city"
        ]
    ).agg(
        orders=("sale_date", "count"),
        revenue=("total_price", "sum")
    ).reset_index()
    monthly["sale_date"] = monthly["sale_date"].astype(str)
    monthly["revenue"] = monthly["revenue"].round(2)
    print(f"[TRANSFORM] {len(monthly)} months processed")
    return monthly

# === LOAD ===
%mkdir -p tmp
def load_report(monthly):
    """Save CSV and chart."""
    ts = datetime.now().strftime("%Y%m%d_%H%M")
    monthly.to_csv(f"tmp/monthly_report_{ts}.csv", index=False)
    px.bar(
        monthly,
        x="sale_date", y="revenue", color="city",
        title=f"UrbanStyle Monthly Revenue ({ts})",
        labels={"sale_date": "Month", "revenue": "Revenue (EUR)", "city": "City"}
    ).write_html(f"tmp/monthly_chart_{ts}.html")
    print(f"[LOAD] CSV + HTML saved")

# === RUN ===
print("PIPELINE START")
df = extract_orders()
monthly = transform_monthly(df)
load_report(monthly)
print("PIPELINE COMPLETE")
print(monthly.to_string(index=False))